In [ ]:
import os
import random
import time
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd
from PIL import Image
from datetime import datetime
import csv
import json
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms, models

import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from jpeg_aug import RandomJPEGCompression
from helper_functions import *

In [ ]:
# Checking Device/confirming it works with CUDA

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

x = torch.randn(10000, 10000, device=device)
print("Computation successful on", device)

## Configuration

In [ ]:
# adjust as needed to main path where data is. Should be 'DS6050_Ai_Detection' folder
data_root = Path(r"C:/Users/Jimmy/OneDrive/Desktop/test/DS6050_Ai_Detection")

# train/validate on percent of data. 0.1 = 10%, 1.0 = 100%
train_percent = 0.2

# adjust as needed for training device
batch_size = 16

# workers seem to be bugged, 0 works best
num_workers = 0

learning_rate = 1e-4

# using jpeg compression augmentation during training?
jpeg_compression = True

# number of epochs per window training step, default is 1 for AI-GenBench dataset
num_epochs_per_step = 1

# Pick any 4 models: resnet50, vit, resnet50_fft, vit_fft
model_name = 'resnet50'

# additional name to add at end of model name for logging and pth file
save_model_name = "baseline"

# fft helper for dataloading
fft = False

## Loading Data

In [ ]:
train_loader, val_loader, train_dataset, val_dataset, csv_name = main_data_loading(data_root, model_name, train_percent,
                                                                                    batch_size, num_workers=num_workers, 
                                                                                    jpeg_compression=jpeg_compression)

## Confirming Window Splits

In [ ]:
train_real_indices, train_fake_indices_by_w, val_real_indices, val_fake_indices_by_w, max_w = confirm_windows(train_dataset, val_dataset)

## Confirm Labels

In [ ]:
confirm_labels(train_dataset, val_dataset)

## Making model

In [ ]:
# MODEL SETUP (ResNet50)

model = models.resnet50(weights=True)
num_ftrs = model.fc.in_features

model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 512),   # hidden layer
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, 2)           # final output
)

model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1) # adding label smoothing for better generalization

optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4) # adding L2 regularization

## Showing JPEG transform

This will show the JPEG transform. Feel free to change the 'index' number to see different images. Will randomly pick a compression amount within the given 'jpeg_q_range'

In [ ]:
show_jpeg_transform_effect(val_dataset, index = 10, jpeg_q_range=(5, 60)) # range can go from 5 - 95

## Create Grid of Validating Images

Will create grid of n images (n = grid_size * grid_size). In order to load images correctly, needs to know if fft is being run, and what device is being used

In [ ]:
create_grid_of_val_images(val_dataset, grid_size=4, fft=fft, device=device)

## Main training loop

Will save the model with the best f1 overall score

In [ ]:
history = sliding_window_training(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    train_real_indices=train_real_indices,
    train_fake_indices_by_w=train_fake_indices_by_w,
    val_real_indices=val_real_indices,
    val_fake_indices_by_w=val_fake_indices_by_w,
    max_w=max_w,
    num_epochs_per_step=num_epochs_per_step,
    batch_size=batch_size,
    device=device,
    model_name=model_name,
    num_workers=num_workers,
    csv_name_used=csv_name,
    model_save_name=save_model_name #not a needed parameter, but will add something at end of model name logger and pth to make it unique if wanted
)